In [1]:
1# This notebook integrates DHS datasets to extract required features for modelling
import duckdb
import pandas as pd
import numpy as np

# Load DTA file
dhs_PKHR = "PSKISTAN/PKHR71FL.DTA"
dhs_PKMR = "PSKISTAN/PKMR71FL.DTA"
dhs_PKIR = "PSKISTAN/PKIR71FL.DTA"

pkhr_use_cols=[
    'hhid','hv000','hv001','hv002', #hv000-country code and phase, hv001-cluster number, hv002-household number
    'hv005', # simpliling weight
    'hv270', # wealth index
    'hv012', #Whether the member is a de jure household member, i.e., whether the member is a usual resident of the household.
    'hv013', #Whether the member is a de facto household member, i.e., whether the member slept in the household the previous night.
     # MEPI indicators 
    'hv206',     # electricity
    'hv226',     # cooking fuel
    'hv241',     # kitchen type food cooked in the house
    'hv242',     # household has separate room used as kitchen yes no
    'hv209',     # refrigerator
    'hv221',     # landline
    'hv243a',    # mobile phone
    'hv208',     # television
    'hv207',     # radio
    'hv213',    #floor material
    'hv215',    #root material

    # socio-demographic variables
    'hv024',    # region division
    'hv025',    # type of place of residence urban/rural
    'hv219',    # sex of head
    'hv220',    # age of head
    'hv106_01', # education of head
    'hv115_01', # marital status
    'hv009',    # family side members of the household
    'hv247',   # bank account
    'hv216',   # household size rooms used for sleeping
    'hv014'   # children under 5 (de jure)
]

pkmr_use_cols = [
'mcaseid', #case identification
'mv000',	#country code and phase
'mv001',	#cluster number
'mv002',	#household number
'mv150',	#relationship to household head
'mv714',	#currently working
'mv717',	#occupation (grouped)
'mv745a', #	owns a house alone or jointly
]

pkir_use_cols = [
'caseid', #case identification
'v000',	#country code and phase
'v001',	#cluster number
'v002',	#household number
'v150',	#relationship to household head
'v714',	#currently working
'v717',	#occupation (grouped)
'v745a'  #owns a house alone or jointly
]

df_PKHR = pd.read_stata(dhs_PKHR, columns=pkhr_use_cols, convert_categoricals=False)

df_PKMR = pd.read_stata(dhs_PKMR, columns=pkmr_use_cols, convert_categoricals=False)

df_PKIR = pd.read_stata(dhs_PKIR, columns=pkir_use_cols,convert_categoricals=False)


# Convert problematic df_HR numpy arrays to regular Python types
for col in df_PKHR.columns:
    if df_PKHR[col].dtype == object:
        # Convert object columns that might contain numpy arrays
        df_PKHR[col] = df_PKHR[col].astype(str)
    elif isinstance(df_PKHR[col].dtype, pd.CategoricalDtype):
        # Convert categorical to string
        df_PKHR[col] = df_PKHR[col].astype(str)
        
# Convert problematic df_MR numpy arrays to regular Python types
for col in df_PKMR.columns:
    if df_PKMR[col].dtype == object:
        # Convert object columns that might contain numpy arrays
        df_PKMR[col] = df_PKMR[col].astype(str)
    elif isinstance(df_PKMR[col].dtype, pd.CategoricalDtype):
        # Convert categorical to string
        df_PKMR[col] = df_PKMR[col].astype(str)

# Convert problematic df_IR numpy arrays to regular Python types
for col in df_PKIR.columns:
    if df_PKIR[col].dtype == object:
        # Convert object columns that might contain numpy arrays
        df_PKIR[col] = df_PKIR[col].astype(str)
    elif isinstance(df_PKIR[col].dtype, pd.CategoricalDtype):
        # Convert categorical to string
        df_PKIR[col] = df_PKIR[col].astype(str)

df_PK_HR_MR_IR = duckdb.sql("""
SELECT 
hhid AS 'hhidCaseIdentification',
hv000 AS 'hv000CountryCode',
hv001 AS 'hv001ClusterNumber',
hv002 AS 'hv002HouseholdNumber',
hv005 AS 'hv005SimplilingWeight',
hv270 AS 'hv270WealthIndex',
hv206 AS 'hv206HasElectricity',
hv226 AS 'hv226CookingFuel',
hv241 AS 'hv241FoodCookedInHouse',
hv242 AS 'hv242SeparateKitchenYesNo',
hv209 AS 'hv209HasRefrigerator',
hv221 AS 'hv221HasLandLine',
hv243a AS 'hv243aHasMobilePhone',
hv208 AS 'hv208HasTelevision',
hv024 AS 'hv024RegionDivision', 
hv025 AS 'hv025TypeOfResidence', 
hv219 AS 'hv219SexOfHead',
hv220 AS 'hv220AgeOfHead',
hv106_01 AS 'hv106_01EducationOfHead',
hv115_01 AS 'hv115_01MaritalStatus',
hv009 AS 'hv009FamilySize',
hv247 AS 'hv247HasBankAccount',
hv216 AS 'hv216HouseSize',  
hv014 AS 'hv014NoOfChildren'
,CASE 
        WHEN HR.hv220 BETWEEN 15 AND 49 AND HR.hv219 = 1 THEN COALESCE(MR.mv714, 0)
        WHEN HR.hv220 BETWEEN 15 AND 49 AND HR.hv219 = 2 THEN COALESCE(IR.v714, 0)
        ELSE 0
    END AS 'v714CurrentlyWorking',
    CASE 
        WHEN HR.hv220 BETWEEN 15 AND 49 AND HR.hv219 = 1 THEN COALESCE(MR.mv717, 0)
        WHEN HR.hv220 BETWEEN 15 AND 49 AND HR.hv219 = 2 THEN COALESCE(IR.v717, 0)
        ELSE 0
    END AS 'v717Occupation',
    CASE 
        WHEN HR.hv220 BETWEEN 15 AND 49 AND HR.hv219 = 1 THEN COALESCE(MR.mv745a, 0)
        WHEN HR.hv220 BETWEEN 15 AND 49 AND HR.hv219 = 2 THEN COALESCE(IR.v745a, 0)
        ELSE 0
    END AS '745aHouseOwnership'
FROM df_PKHR HR 
LEFT JOIN df_PKMR MR 
    ON HR.hv001 = MR.mv001 
    AND HR.hv002 = MR.mv002
    AND MR.mv150 = 1  -- Man is household head
    AND HR.hv219 = 1  -- Head is male
LEFT JOIN df_PKIR IR 
    ON HR.hv001 = IR.v001 
    AND HR.hv002 = IR.v002
    AND IR.v150 = 1   -- Woman is household head
    AND HR.hv219 = 2  -- Head is female
""").df()

df_PK_HR_MR_IR.head()


,hhidCaseIdentification,hv000CountryCode,hv001ClusterNumber,hv002HouseholdNumber,hv005SimplilingWeight,hv270WealthIndex,hv206HasElectricity,hv226CookingFuel,hv241FoodCookedInHouse,hv242SeparateKitchenYesNo,...,hv220AgeOfHead,hv106_01EducationOfHead,hv115_01MaritalStatus,hv009FamilySize,hv247HasBankAccount,hv216HouseSize,hv014NoOfChildren,v714CurrentlyWorking,v717Occupation,745aHouseOwnership
0,1 1,PK7,1,1,849785,1,1.0,8.0,1.0,0.0,...,44,2.0,1.0,7,0.0,1.0,1,1.0,9.0,1.0
1,1 7,PK7,1,7,849785,1,1.0,8.0,1.0,0.0,...,43,2.0,1.0,7,1.0,1.0,0,1.0,9.0,0.0
2,1 10,PK7,1,10,849785,2,1.0,8.0,1.0,0.0,...,45,0.0,1.0,11,1.0,2.0,1,1.0,8.0,2.0
3,2 1,PK7,2,1,72885,3,1.0,8.0,1.0,1.0,...,32,2.0,1.0,9,1.0,2.0,2,1.0,8.0,0.0
4,2 4,PK7,2,4,72885,4,1.0,8.0,1.0,1.0,...,43,3.0,1.0,5,1.0,2.0,0,1.0,1.0,1.0


In [2]:
df_PK_HR_MR_IR["weight"] = df_PK_HR_MR_IR["hv005SimplilingWeight"] / 1_000_000

#save data into parquet
df_PK_HR_MR_IR.to_parquet(
    "PAKISTAN/DATA/1.df_PK_NecessaryColumns.parquet",
    engine="pyarrow",
    compression="snappy"
)
print(f"✅ File saved.")

✅ File saved.
